# <center>XGBoost por Subconjunto de Jets usando la Mezcla i3 + tree_safe</center>


Este notebook toma el mismo preprocesador de `11.Mezcla/01`, pero lo ajusta por separado en cuatro subconjuntos definidos por `PRI_jet_num = 0, 1, 2, 3`. Luego entrena cuatro `XGBClassifier` independientes y hace el ruteo de inferencia segun la cantidad de jets del evento.


## Imports y utilidades


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError("No se pudo importar xgboost. Instalarlo antes de ejecutar este notebook.") from exc


def find_mix_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct_candidate = candidate / "i3_tree_safe_mix_by_jets.py"
        nested_candidate = candidate / "11.Mezcla" / "i3_tree_safe_mix_by_jets.py"
        if direct_candidate.exists():
            return candidate
        if nested_candidate.exists():
            return candidate / "11.Mezcla"
    raise FileNotFoundError("No se pudo ubicar 11.Mezcla/i3_tree_safe_mix_by_jets.py")


mix_dir = find_mix_dir()
if str(mix_dir) not in sys.path:
    sys.path.insert(0, str(mix_dir))

from i3_tree_safe_mix import find_project_root, get_higgs_data_dir
from i3_tree_safe_mix_by_jets import (
    JET_GROUPS,
    run_i3_tree_safe_mix_by_jets,
    save_i3_tree_safe_mix_by_jets_artifacts,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
SEED = 42
BEST_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": SEED,
    "learning_rate": 0.05,
    "n_estimators": 400,
    "max_depth": 6,
    "scale_pos_weight": 1.5346,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "tree_method": "hist",
    "n_jobs": -1,
    "missing": np.nan,
}


## Estrategia
1. Hacer un split global `train/valid` antes de cualquier ajuste.
2. Dentro de cada split, partir las filas por `PRI_jet_num`.
3. Ajustar un preprocesador `i3_tree_safe_mix` por grupo.
4. Entrenar un `XGBClassifier` por grupo.
5. Para inferencia, enrutar cada fila al pipeline y modelo correspondiente por cantidad de jets.


In [ ]:
artifacts = run_i3_tree_safe_mix_by_jets()
output_dir = get_higgs_data_dir() / "processed" / "i3_tree_safe_mix_by_jets"
metadata = artifacts["metadata"]

print("Variante: Mezcla i3 + tree_safe separada por jets")
print(f"Directorio de salida esperado: {output_dir}")
print("Descripcion:")
print(metadata["description"])


## Resumen de la preparación por jets


In [ ]:
print("Resumen global del split temprano:")
display(artifacts["split_summary_df"].round(4))

print("Resumen por grupo de jets:")
display(artifacts["group_summary_df"].round(4))

print("Metadata principal:")
for key, value in metadata.items():
    print(f"- {key}: {value}")


## Entrenamiento de 4 modelos y evaluación consolidada


In [ ]:
def ams_score(signal_weight, background_weight, br: float = 10.0):
    signal_arr = np.asarray(signal_weight, dtype=np.float64)
    background_arr = np.asarray(background_weight, dtype=np.float64)
    radicand = 2.0 * (
        (signal_arr + background_arr + br) * np.log1p(signal_arr / (background_arr + br)) - signal_arr
    )
    return np.sqrt(np.clip(radicand, a_min=0.0, a_max=None))


def find_best_ams_threshold(y_true, y_scores, weights, br: float = 10.0) -> dict[str, float | int]:
    y_array = np.asarray(y_true, dtype=np.int8)
    score_array = np.asarray(y_scores, dtype=np.float64)
    weight_array = np.asarray(weights, dtype=np.float64)

    order = np.argsort(score_array)[::-1]
    y_sorted = y_array[order]
    score_sorted = score_array[order]
    weight_sorted = weight_array[order]

    signal_cumsum = np.cumsum(np.where(y_sorted == 1, weight_sorted, 0.0))
    background_cumsum = np.cumsum(np.where(y_sorted == 0, weight_sorted, 0.0))

    candidate_mask = np.r_[score_sorted[1:] != score_sorted[:-1], True]
    candidate_indices = np.flatnonzero(candidate_mask)
    ams_values = ams_score(signal_cumsum[candidate_indices], background_cumsum[candidate_indices], br=br)

    best_position = int(np.argmax(ams_values))
    best_idx = int(candidate_indices[best_position])
    best_threshold = float(score_sorted[best_idx])
    best_pred = (score_array >= best_threshold).astype(np.int8)

    return {
        "threshold": best_threshold,
        "ams": float(ams_values[best_position]),
        "signal_weight": float(signal_cumsum[best_idx]),
        "background_weight": float(background_cumsum[best_idx]),
        "predicted_signal_count": int(best_pred.sum()),
    }


def fit_group_model(group: dict[str, object]) -> tuple[XGBClassifier, dict[str, float | int]]:
    X_train = group["X_train_processed"]
    X_valid = group["X_valid_processed"]
    y_train = group["y_train"].map({"b": 0, "s": 1}).astype(np.int8)
    y_valid = group["y_valid"].map({"b": 0, "s": 1}).astype(np.int8)
    valid_weight = group["valid_weight"].astype(np.float64)

    model = XGBClassifier(**BEST_XGB_PARAMS)
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start

    valid_score = model.predict_proba(X_valid)[:, 1]
    valid_pred = model.predict(X_valid)
    ams_info = find_best_ams_threshold(y_valid, valid_score, valid_weight)

    metrics = {
        "jet_num": int(group["jet_value"]),
        "train_rows": int(len(X_train)),
        "valid_rows": int(len(X_valid)),
        "feature_count": int(X_train.shape[1]),
        "train_nan_count": int(X_train.isna().sum().sum()),
        "valid_nan_count": int(X_valid.isna().sum().sum()),
        "accuracy": float(accuracy_score(y_valid, valid_pred)),
        "precision": float(precision_score(y_valid, valid_pred, zero_division=0)),
        "recall": float(recall_score(y_valid, valid_pred, zero_division=0)),
        "f1": float(f1_score(y_valid, valid_pred, zero_division=0)),
        "auc": float(roc_auc_score(y_valid, valid_score)),
        "ams": float(ams_info["ams"]),
        "ams_threshold": float(ams_info["threshold"]),
        "fit_seconds": float(fit_seconds),
    }
    return model, metrics


group_models = {}
group_metrics_rows = []
valid_prediction_parts = []

for jet_value in JET_GROUPS:
    group = artifacts["group_artifacts"][jet_value]
    model, metrics = fit_group_model(group)
    group_models[jet_value] = model
    group_metrics_rows.append(metrics)

    valid_scores = model.predict_proba(group["X_valid_processed"])[:, 1]
    valid_prediction_parts.append(
        pd.DataFrame(
            {
                "EventId": group["valid_event_id"].astype(np.int64),
                "y_true": group["y_valid"].map({"b": 0, "s": 1}).astype(np.int8),
                "weight": group["valid_weight"].astype(np.float64),
                "score": valid_scores,
                "jet_num": jet_value,
            }
        )
    )

group_metrics_df = pd.DataFrame(group_metrics_rows).sort_values("jet_num").reset_index(drop=True)
print("Metricas por modelo especializado en cada jet:")
display(group_metrics_df.round(6))

valid_predictions_df = pd.concat(valid_prediction_parts, ignore_index=True)
valid_order = artifacts["global_valid_event_id"].astype(np.int64)
valid_predictions_df = (
    valid_predictions_df.set_index("EventId").loc[valid_order].reset_index()
)

best_global_threshold = find_best_ams_threshold(
    valid_predictions_df["y_true"],
    valid_predictions_df["score"],
    valid_predictions_df["weight"],
)
valid_pred_global = (valid_predictions_df["score"].to_numpy() >= best_global_threshold["threshold"]).astype(np.int8)

overall_metrics_df = pd.DataFrame(
    [
        {
            "dataset": "xgboost_i3_tree_safe_mix_by_jets",
            "valid_rows": int(len(valid_predictions_df)),
            "accuracy": float(accuracy_score(valid_predictions_df["y_true"], valid_pred_global)),
            "precision": float(precision_score(valid_predictions_df["y_true"], valid_pred_global, zero_division=0)),
            "recall": float(recall_score(valid_predictions_df["y_true"], valid_pred_global, zero_division=0)),
            "f1": float(f1_score(valid_predictions_df["y_true"], valid_pred_global, zero_division=0)),
            "auc": float(roc_auc_score(valid_predictions_df["y_true"], valid_predictions_df["score"])),
            "ams": float(best_global_threshold["ams"]),
            "ams_threshold": float(best_global_threshold["threshold"]),
            "predicted_signal_count": int(valid_pred_global.sum()),
        }
    ]
)

print("Metrica global despues de enrutar por PRI_jet_num:")
display(overall_metrics_df.round(6))


## Ejemplo de inferencia enrutada sobre test


In [ ]:
test_prediction_parts = []
for jet_value in JET_GROUPS:
    group = artifacts["group_artifacts"][jet_value]
    model = group_models[jet_value]
    test_scores = model.predict_proba(group["X_test_processed"])[:, 1]
    test_prediction_parts.append(
        pd.DataFrame(
            {
                "EventId": group["test_event_id"].astype(np.int64),
                "score": test_scores,
                "jet_num": jet_value,
            }
        )
    )

test_scores_df = pd.concat(test_prediction_parts, ignore_index=True)
test_order = artifacts["global_test_event_id"].astype(np.int64)
test_scores_df = test_scores_df.set_index("EventId").loc[test_order].reset_index()

print("Primeras filas del enrutamiento de inferencia sobre test:")
display(test_scores_df.head())
print(f"Filas totales reconstruidas: {len(test_scores_df)}")


## Guardado de artefactos
Se guardan los datasets procesados y el pipeline ajustado para cada grupo de jets.


In [ ]:
saved_paths = save_i3_tree_safe_mix_by_jets_artifacts(artifacts, output_dir=output_dir)

print(f"Directorio de salida: {output_dir}")
print(f"Cantidad de artefactos guardados: {len(saved_paths)}")
for path in saved_paths[:20]:
    print(f"- {path}")
if len(saved_paths) > 20:
    print(f"... y {len(saved_paths) - 20} artefactos adicionales")
